# Battery Model Dispatch Plot

This notebook loads the electricity price data and plots RL battery dispatch against mathematical optimization together with the electricity price.

In [1]:
import os
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"  # must be before pyplot import

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "Battery_model").exists():
            return candidate
    raise FileNotFoundError("Could not find the THESIS repository root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PRICE_FILE = (
    ROOT
    / "Battery_model"
    / "Optimization_Pyomo"
    / "input_data"
    / "electricity_price.txt"
)
RL_MODEL_PATH = ROOT / "src" / "battery_model" / "battery_sac_model"
OUTPUT_FIGURE = ROOT / "notebooks" / "battery_model_dispatch_plot.png"

# Use one day for the dispatch comparison plot.
WINDOW_START = 120
WINDOW_HOURS = 24

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

ROOT


PosixPath('/home/ashish/Documents/THESIS')

In [2]:
price_df = pd.read_csv(
    PRICE_FILE,
    sep=r"\s+",
    header=None,
    names=["timestamp_s", "price_eur_per_mwh"],
)

prices_full = price_df["price_eur_per_mwh"].astype(float).to_numpy()
GLOBAL_PRICE_SCALE = np.percentile(np.abs(prices_full), 95) + 1e-6

window_end = min(WINDOW_START + WINDOW_HOURS, len(prices_full))
prices_window = prices_full[WINDOW_START:window_end]

print(f"Loaded {len(prices_full)} hourly prices from {PRICE_FILE.relative_to(ROOT)}")
print(f"Using hours {WINDOW_START} to {window_end - 1} ({len(prices_window)} hours)")
print(f"Global price scale: {GLOBAL_PRICE_SCALE:.4f}")

price_df.iloc[WINDOW_START:window_end].head()


Loaded 8784 hourly prices from Battery_model/Optimization_Pyomo/input_data/electricity_price.txt
Using hours 120 to 143 (24 hours)
Global price scale: 65.5685


,timestamp_s,price_eur_per_mwh
120,432000,14.88
121,435600,11.72
122,439200,6.29
123,442800,10.19
124,446400,10.98


In [3]:
def build_dispatch_frame(
    method, prices, power_kw, soc, power_actual_kw=None, control=None, rewards=None
):
    prices = np.asarray(prices, dtype=float)
    power_kw = np.asarray(power_kw, dtype=float)
    soc = np.asarray(soc, dtype=float)

    data = {
        "hour": np.arange(len(prices)),
        "price_eur_per_mwh": prices,
        "power_kw": power_kw,
        "charge_kw": np.clip(power_kw, 0.0, None),
        "discharge_kw": np.clip(-power_kw, 0.0, None),
        "soc": soc,
        "method": method,
    }
    if power_actual_kw is not None:
        data["power_actual_kw"] = np.asarray(power_actual_kw, dtype=float)
    if control is not None:
        data["control_u"] = np.asarray(control, dtype=float)
    if rewards is not None:
        data["reward"] = np.asarray(rewards, dtype=float)

    return pd.DataFrame(data)


def run_rl_dispatch(prices, price_scale):
    from stable_baselines3 import SAC

    from src.battery_model.evaluate import evaluate_model

    if not RL_MODEL_PATH.with_suffix(".zip").exists():
        raise FileNotFoundError(
            f"Missing saved RL model: {RL_MODEL_PATH.with_suffix('.zip')}"
        )

    model = SAC.load(str(RL_MODEL_PATH))
    p_cmds, power_kw, power_actual_kw, soc, rewards, prices_used = evaluate_model(
        model,
        prices,
        price_scale,
    )
    return build_dispatch_frame(
        "RL SAC",
        prices_used,
        power_kw,
        soc,
        power_actual_kw=power_actual_kw,
        control=p_cmds / 20.0,
        rewards=rewards,
    )


def run_pyomo_dispatch(prices):
    from Battery_model.Optimization_Pyomo.pyomo_battery_model_v2 import (
        get_battery_model,
    )

    result = get_battery_model(prices, verbose=False)
    if result.get("status") != "success":
        raise RuntimeError(result.get("error", "Pyomo optimization failed."))

    return build_dispatch_frame(
        "Mathematical optimization",
        prices,
        result["P"],
        result["SOC"],
        power_actual_kw=result.get("P_actual"),
        control=result.get("u"),
    )


def objective_value_eur(df):
    return float(np.sum((df["price_eur_per_mwh"] / 1000.0) * df["power_kw"]))


def summarize_dispatch(df):
    objective_eur = objective_value_eur(df)
    return pd.DataFrame(
        [
            {
                "method": df["method"].iloc[0],
                "window_hours": len(df),
                "objective_value_eur": objective_eur,
                "total_cost_eur": objective_eur,
                "charge_energy_kwh": float(df["charge_kw"].sum()),
                "discharge_energy_kwh": float(df["discharge_kw"].sum()),
                "min_soc": float(df["soc"].min()),
                "max_soc": float(df["soc"].max()),
            }
        ]
    )


In [4]:
dispatch_df = run_rl_dispatch(prices_window, GLOBAL_PRICE_SCALE)
math_dispatch_df = run_pyomo_dispatch(prices_window)

print("Prepared RL SAC dispatch for the plot.")
print("Prepared mathematical optimization dispatch for the plot.")

summary_frames = [
    summarize_dispatch(dispatch_df),
    summarize_dispatch(math_dispatch_df),
]

display(pd.concat(summary_frames, ignore_index=True))
dispatch_df.head()


Prepared RL SAC dispatch for the plot.
Prepared mathematical optimization dispatch for the plot.


,method,window_hours,objective_value_eur,total_cost_eur,charge_energy_kwh,discharge_energy_kwh,min_soc,max_soc
0,RL SAC,24,-3.287507,-3.287507,122.804972,96.124194,0.518986,0.999381
1,Mathematical optimization,24,-3.689975,-3.689975,111.893908,90.799919,0.497278,0.999151


,hour,price_eur_per_mwh,power_kw,charge_kw,discharge_kw,soc,method,power_actual_kw,control_u,reward
0,0,14.88,18.250337,18.250337,0.0,0.582127,RL SAC,16.425303,0.912517,-0.271565
1,1,11.72,18.687382,18.687382,0.0,0.666220,RL SAC,16.818644,0.934369,-0.219016
2,2,6.29,18.979721,18.979721,0.0,0.751628,RL SAC,17.081749,0.948986,-0.119382
3,3,10.19,18.682623,18.682623,0.0,0.835700,RL SAC,16.814361,0.934131,-0.190376
4,4,10.98,18.436117,18.436117,0.0,0.918663,RL SAC,16.592505,0.921806,-0.202429


In [5]:
def plot_battery_dispatch(df, math_df, output_path=None):
    method = df["method"].iloc[0]
    hours = df["hour"].to_numpy()
    x_min, x_max = 0, 23
    x_ticks = np.arange(x_min, x_max + 1, 1)
    primary_label = "RL" if method == "RL SAC" else method

    fig, (ax_power, ax_soc) = plt.subplots(
        2,
        1,
        figsize=(14, 5.2),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.08},
    )

    ax_power.fill_between(
        hours,
        0,
        df["charge_kw"],
        step="post",
        color="#7777ff",
        alpha=0.45,
        label=f"{primary_label} charge",
    )
    ax_power.fill_between(
        hours,
        0,
        df["discharge_kw"],
        step="post",
        color="#79bd7a",
        alpha=0.50,
        label=f"{primary_label} discharge",
    )
    ax_power.step(hours, df["charge_kw"], where="post", color="#3434a8", linewidth=1.1)
    ax_power.step(
        hours, df["discharge_kw"], where="post", color="#276b35", linewidth=1.1
    )

    max_power = max(float(df["charge_kw"].max()), float(df["discharge_kw"].max()), 1.0)
    if math_df is not None and math_df is not df:
        math_hours = math_df["hour"].to_numpy()
        ax_power.step(
            math_hours,
            math_df["charge_kw"],
            where="post",
            color="#0000c7",
            linewidth=2.0,
            linestyle="--",
            label="Math charge",
        )
        ax_power.step(
            math_hours,
            math_df["discharge_kw"],
            where="post",
            color="#0b5d1e",
            linewidth=2.0,
            linestyle="--",
            label="Math discharge",
        )
        max_power = max(
            max_power,
            float(math_df["charge_kw"].max()),
            float(math_df["discharge_kw"].max()),
        )
    ax_power.set_ylim(0, max_power * 1.25)
    ax_power.set_xlim(x_min, x_max)
    ax_power.set_ylabel("Power [kW]", fontweight="bold")
    ax_power.grid(True, alpha=0.28)
    ax_power.tick_params(labelbottom=False)

    ax_price = ax_power.twinx()
    ax_price.plot(
        hours,
        df["price_eur_per_mwh"],
        color="red",
        linewidth=2.0,
        label="Electricity price",
    )
    ax_price.set_ylabel("Electricity price [EUR/MWh]", fontweight="bold")

    handles_1, labels_1 = ax_power.get_legend_handles_labels()
    handles_2, labels_2 = ax_price.get_legend_handles_labels()
    ax_power.legend(
        handles_1 + handles_2,
        labels_1 + labels_2,
        loc="upper left",
        frameon=True,
        ncol=3,
    )

    objective_lines = [f"{primary_label} objective: {objective_value_eur(df):.2f} EUR"]
    if math_df is not None and math_df is not df:
        objective_lines.append(
            f"Math objective: {objective_value_eur(math_df):.2f} EUR"
        )
    ax_power.text(
        0.985,
        0.06,
        "\n".join(objective_lines),
        transform=ax_power.transAxes,
        ha="right",
        va="bottom",
        fontsize=9,
        bbox={
            "facecolor": "white",
            "edgecolor": "#bbbbbb",
            "alpha": 0.88,
            "boxstyle": "round,pad=0.35",
        },
    )

    ax_soc.plot(
        hours,
        df["soc"] * 100.0,
        color="#1f1f1f",
        linewidth=1.9,
        label=f"{primary_label} SOC",
    )
    ax_soc.fill_between(hours, 0, df["soc"] * 100.0, color="#1f1f1f", alpha=0.08)
    if math_df is not None and math_df is not df:
        ax_soc.plot(
            math_df["hour"],
            math_df["soc"] * 100.0,
            color="#555555",
            linewidth=1.8,
            linestyle="--",
            label="Math SOC",
        )
    ax_soc.set_ylim(0, 100)
    ax_soc.set_xlim(x_min, x_max)
    ax_soc.set_xticks(x_ticks)
    ax_soc.set_xlabel("Time [h]")
    ax_soc.set_ylabel("SOC [%]", fontweight="bold")
    ax_soc.grid(True, alpha=0.28)
    ax_soc.legend(loc="upper left", frameon=True)

    if math_df is not None and math_df is not df:
        title = f"Battery dispatch: {primary_label} vs mathematical optimization | hours {WINDOW_START}-{window_end - 1}"
    else:
        title = (
            f"Battery dispatch from {method} | hours {WINDOW_START}-{window_end - 1}"
        )
    ax_power.set_title(title)
    fig.subplots_adjust(top=0.90)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {output_path.relative_to(ROOT)}")

    return fig


plot_battery_dispatch(dispatch_df, math_dispatch_df, OUTPUT_FIGURE)

Saved figure to notebooks/battery_model_dispatch_plot.png


<Figure size 1680x624 with 3 Axes>

The plot always compares RL SAC against the mathematical optimizer. Pyomo requires a working IPOPT installation.